# 🎲 Module 3.1: Probability Theory for Reinforcement Learning

## The Mathematical Language of Uncertainty and Decision Making

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FlashVision/VisionRL/blob/main/Module_03_RL_Mathematical_Foundations/01_Probability_For_RL/probability_for_rl.ipynb)

---

## 🎯 Learning Objectives
1. Understand probability spaces and their axioms
2. Work with random variables, expectations, and variance
3. Apply conditional probability and Bayes' theorem
4. Understand the Markov property and its significance for RL
5. Connect probability concepts to image pixel distributions

---

In [ ]:
# Setup - Install dependencies (Google Colab compatible)
!pip install numpy matplotlib scipy scikit-image pillow -q

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from skimage import data, color
import warnings
warnings.filterwarnings('ignore')

plt.style.use('default')
np.random.seed(42)
print("✅ All libraries loaded successfully!")

In [ ]:
# Download REAL open-source dataset for RL demonstrations
import torchvision
import numpy as np

# MNIST as discrete image states for MDP demonstrations
mnist = torchvision.datasets.MNIST(root='./data', train=True, download=True)
# Use small subset as "states" in our MDP
sample_states = [np.array(mnist[i][0]) for i in range(100)]
print(f"✅ MNIST loaded: Using {len(sample_states)} real images as MDP states")
print("   Each state is a 28x28 grayscale image = 784-dimensional state vector")

## Deep Derivation: From Axioms to Markov Property

### Step 1: Kolmogorov Axioms
1. $P(\Omega) = 1$ (certainty)
2. $P(A) \geq 0$ for all events $A$ (non-negativity)
3. $P(\bigcup_{i=1}^\infty A_i) = \sum_{i=1}^\infty P(A_i)$ for disjoint $A_i$ (countable additivity)

### Step 2: Deriving Bayes' Theorem (Full Proof)
**Start:** Definition of conditional probability: $P(A|B) = \frac{P(A \cap B)}{P(B)}$

**Step 2a:** By symmetry: $P(B|A) = \frac{P(A \cap B)}{P(A)}$

**Step 2b:** Solve for $P(A \cap B)$: $P(A \cap B) = P(B|A) \cdot P(A)$

**Step 2c:** Substitute into Step 2a:
$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)} \quad \blacksquare$$

**Step 2d:** Expand denominator using law of total probability:
$$P(B) = \sum_{i} P(B|A_i) P(A_i)$$

**Full Bayes:**
$$P(A_i|B) = \frac{P(B|A_i) P(A_i)}{\sum_j P(B|A_j) P(A_j)}$$

### Step 3: Expectation Properties (Proofs)

**Linearity:** $E[aX + bY] = aE[X] + bE[Y]$

**Proof:**
$$E[aX + bY] = \sum_x \sum_y (ax + by) P(X=x, Y=y)$$
$$= a\sum_x x \sum_y P(X=x, Y=y) + b\sum_y y \sum_x P(X=x, Y=y)$$
$$= a\sum_x x \cdot P(X=x) + b\sum_y y \cdot P(Y=y) = aE[X] + bE[Y] \quad \blacksquare$$

**Tower Property:** $E[E[X|Y]] = E[X]$

**Proof:**
$$E[E[X|Y]] = \sum_y E[X|Y=y] \cdot P(Y=y) = \sum_y \left(\sum_x x \cdot P(X=x|Y=y)\right) P(Y=y)$$
$$= \sum_x \sum_y x \cdot P(X=x|Y=y) \cdot P(Y=y) = \sum_x x \cdot P(X=x) = E[X] \quad \blacksquare$$

### Step 4: Markov Property (Formal Definition + Proof of Memorylessness)
A process $\{X_t\}$ is Markov if:
$$P(X_{t+1} = x | X_t = x_t, X_{t-1} = x_{t-1}, \ldots, X_0 = x_0) = P(X_{t+1} = x | X_t = x_t)$$

**Why images satisfy this for RL:**
- Current image $I_t$ contains ALL information needed to decide the next action
- Previous images $I_0, \ldots, I_{t-1}$ add no extra information
- The image IS the sufficient statistic: $I_t \perp\!\!\!\perp (I_0, \ldots, I_{t-2}) \mid I_{t-1}$

---

## 📐 Section 1: Probability Spaces

A **probability space** is a triple $(\Omega, \mathcal{F}, P)$ where:

| Component | Name | Description |
|-----------|------|-------------|
| $\Omega$ | Sample Space | Set of all possible outcomes |
| $\mathcal{F}$ | Event Space | Collection of subsets of $\Omega$ (a $\sigma$-algebra) |
| $P$ | Probability Measure | Function $P: \mathcal{F} \to [0, 1]$ |

### Kolmogorov Axioms

1. **Non-negativity:** $P(A) \geq 0$ for all $A \in \mathcal{F}$
2. **Normalization:** $P(\Omega) = 1$
3. **Countable Additivity:** For mutually exclusive events $A_1, A_2, \ldots$:

$$P\left(\bigcup_{i=1}^{\infty} A_i\right) = \sum_{i=1}^{\infty} P(A_i)$$

### 🖼️ Image Processing Example

For a grayscale image, we can define:
- **Sample space:** $\Omega = \{0, 1, 2, \ldots, 255\}$ (all possible pixel intensities)
- **Event space:** All subsets of $\Omega$ (e.g., "dark pixels" = $\{0, 1, \ldots, 50\}$)
- **Probability measure:** The normalized histogram $P(k) = \frac{\text{count of pixels with value } k}{\text{total pixels}}$

In [ ]:
# Section 1: Probability Space for Pixel Intensities
# Create a synthetic image with gaussian blobs

def create_gaussian_blobs(size=256):
    """Create a synthetic image with multiple gaussian blobs."""
    image = np.zeros((size, size), dtype=np.float64)
    y, x = np.mgrid[0:size, 0:size]
    
    blob_params = [
        (64, 64, 20, 200),
        (192, 192, 30, 180),
        (64, 192, 25, 220),
        (192, 64, 15, 160),
        (128, 128, 40, 140),
    ]
    
    for cx, cy, sigma, amplitude in blob_params:
        blob = amplitude * np.exp(-((x - cx)**2 + (y - cy)**2) / (2 * sigma**2))
        image += blob
    
    image += np.random.normal(20, 10, (size, size))
    image = np.clip(image, 0, 255).astype(np.uint8)
    return image

image = create_gaussian_blobs()

# Compute the empirical probability distribution (normalized histogram)
hist, bin_edges = np.histogram(image, bins=256, range=(0, 255))
prob_distribution = hist / hist.sum()

# Verify axioms
print("=" * 60)
print("PROBABILITY SPACE FOR PIXEL INTENSITIES")
print("=" * 60)
print("\nSample Space \u03a9 = {0, 1, 2, ..., 255}")
print("Number of outcomes: |\u03a9| = 256")
print("\nVerifying Kolmogorov Axioms:")
print(f"  Axiom 1 (Non-negativity): min P(k) = {prob_distribution.min():.6f} \u2265 0 \u2713")
print(f"  Axiom 2 (Normalization):  \u03a3 P(k) = {prob_distribution.sum():.10f} = 1 \u2713")
print("  Axiom 3 (Additivity):     P(dark) + P(bright) computed below")

# Demonstrate additivity
p_dark = prob_distribution[:51].sum()  # P(pixel in {0,...,50})
p_mid = prob_distribution[51:200].sum()  # P(pixel in {51,...,199})
p_bright = prob_distribution[200:].sum()  # P(pixel in {200,...,255})
print(f"\n  P(dark [0-50])   = {p_dark:.4f}")
print(f"  P(mid [51-199])  = {p_mid:.4f}")
print(f"  P(bright [200+]) = {p_bright:.4f}")
print(f"  Sum              = {p_dark + p_mid + p_bright:.4f} (= 1, additivity holds) \u2713")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(image, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Synthetic Image (Gaussian Blobs)', fontsize=11)
axes[0].axis('off')

axes[1].bar(range(256), prob_distribution, width=1.0, color='steelblue', alpha=0.7)
axes[1].set_xlabel('Pixel Intensity (k)')
axes[1].set_ylabel('P(X = k)')
axes[1].set_title('Empirical Probability Distribution', fontsize=11)
axes[1].set_xlim(0, 255)

axes[2].fill_between(range(51), prob_distribution[:51], alpha=0.5, color='navy', label=f'Dark: P={p_dark:.3f}')
axes[2].fill_between(range(51, 200), prob_distribution[51:200], alpha=0.5, color='green', label=f'Mid: P={p_mid:.3f}')
axes[2].fill_between(range(200, 256), prob_distribution[200:], alpha=0.5, color='red', label=f'Bright: P={p_bright:.3f}')
axes[2].set_xlabel('Pixel Intensity (k)')
axes[2].set_ylabel('P(X = k)')
axes[2].set_title('Partitioned Events (Additivity)', fontsize=11)
axes[2].legend()
axes[2].set_xlim(0, 255)

plt.tight_layout()
plt.show()

---

## 📊 Section 2: Random Variables and Expectations

### Definition

A **random variable** is a measurable function from the sample space to the real numbers:

$$X: \Omega \to \mathbb{R}$$

### Probability Mass Function (PMF)

For a discrete random variable:

$$p(x) = P(X = x)$$

### Expectation (Mean)

The expected value represents the "center of mass" of the distribution:

**Discrete:** $E[X] = \sum_{x} x \cdot p(x)$

**Continuous:** $E[X] = \int_{-\infty}^{\infty} x \, f(x) \, dx$

### Variance

Measures the spread of the distribution:

$$\text{Var}(X) = E[(X - E[X])^2] = E[X^2] - (E[X])^2$$

$$\text{Std}(X) = \sigma = \sqrt{\text{Var}(X)}$$

### 🖼️ For Images
- $E[X]$ = average pixel intensity (overall brightness)
- $\text{Var}(X)$ = how much pixel values vary (contrast indicator)
- Low variance → flat/uniform image, High variance → high contrast image

In [ ]:
# Section 2: Computing E[X] and Var(X) from an image

# Load a real image from skimage
try:
    img = data.camera()  # Classic cameraman image (grayscale)
except Exception:
    img = create_gaussian_blobs(256)

# Flatten to treat as a collection of random variable observations
pixels = img.flatten().astype(np.float64)
n_pixels = len(pixels)

# Compute PMF
values, counts = np.unique(img, return_counts=True)
pmf = counts / n_pixels

# Manual computation of E[X]
E_X_manual = np.sum(values * pmf)

# Manual computation of E[X^2]
E_X2_manual = np.sum(values**2 * pmf)

# Variance using the formula Var(X) = E[X^2] - (E[X])^2
Var_X_manual = E_X2_manual - E_X_manual**2
Std_X_manual = np.sqrt(Var_X_manual)

# Numpy verification
E_X_numpy = np.mean(pixels)
Var_X_numpy = np.var(pixels)
Std_X_numpy = np.std(pixels)

print("=" * 60)
print("RANDOM VARIABLE: Pixel Intensity X")
print("=" * 60)
print(f"\nImage size: {img.shape[0]} x {img.shape[1]} = {n_pixels} pixels")
print(f"\n{'Statistic':<20} {'Manual':>12} {'NumPy':>12} {'Match':>8}")
print("-" * 55)
print(f"{'E[X]':<20} {E_X_manual:>12.4f} {E_X_numpy:>12.4f} {'\u2713':>8}")
print(f"{'Var(X)':<20} {Var_X_manual:>12.4f} {Var_X_numpy:>12.4f} {'\u2713':>8}")
print(f"{'Std(X)':<20} {Std_X_manual:>12.4f} {Std_X_numpy:>12.4f} {'\u2713':>8}")
print("\nInterpretation:")
print(f"  Mean brightness: {E_X_manual:.1f} / 255 = {E_X_manual/255:.1%} of max")
print(f"  Std deviation:   {Std_X_manual:.1f} (higher = more contrast)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'Image (E[X]={E_X_manual:.1f}, \u03c3={Std_X_manual:.1f})', fontsize=11)
axes[0].axis('off')

axes[1].hist(pixels, bins=256, range=(0, 255), density=True, alpha=0.7, color='steelblue')
axes[1].axvline(E_X_manual, color='red', linewidth=2, linestyle='-', label=f'E[X] = {E_X_manual:.1f}')
axes[1].axvline(E_X_manual - Std_X_manual, color='orange', linewidth=1.5, linestyle='--',
               label=f'E[X] - \u03c3 = {E_X_manual - Std_X_manual:.1f}')
axes[1].axvline(E_X_manual + Std_X_manual, color='orange', linewidth=1.5, linestyle='--',
               label=f'E[X] + \u03c3 = {E_X_manual + Std_X_manual:.1f}')
axes[1].fill_betweenx([0, axes[1].get_ylim()[1] if axes[1].get_ylim()[1] > 0 else 0.02],
                      E_X_manual - Std_X_manual, E_X_manual + Std_X_manual,
                      alpha=0.15, color='orange')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Probability Density')
axes[1].set_title('Distribution with Mean and Std', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].set_xlim(0, 255)

plt.tight_layout()
plt.show()

In [ ]:
# Section 2 (continued): Law of Large Numbers Demonstration
# As we sample more pixels, the sample mean converges to the true mean

true_mean = np.mean(pixels)
sample_sizes = np.logspace(1, np.log10(n_pixels), 200).astype(int)
sample_sizes = np.unique(sample_sizes)

sample_means = []
sample_stds = []

np.random.seed(42)
for n in sample_sizes:
    sample = np.random.choice(pixels, size=n, replace=False)
    sample_means.append(np.mean(sample))
    sample_stds.append(np.std(sample) / np.sqrt(n))  # Standard error

sample_means = np.array(sample_means)
sample_stds = np.array(sample_stds)

print("=" * 60)
print("LAW OF LARGE NUMBERS: Sample Mean Convergence")
print("=" * 60)
print(f"\nTrue mean (all {n_pixels} pixels): {true_mean:.4f}")
print(f"\n{'Sample Size':<15} {'Sample Mean':>12} {'Error':>10}")
print("-" * 40)
for n_check in [10, 50, 100, 500, 1000, 5000, n_pixels]:
    idx = np.argmin(np.abs(sample_sizes - n_check))
    print(f"{sample_sizes[idx]:<15} {sample_means[idx]:>12.4f} {abs(sample_means[idx] - true_mean):>10.4f}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].semilogx(sample_sizes, sample_means, 'b-', alpha=0.7, linewidth=1)
axes[0].axhline(true_mean, color='red', linestyle='--', linewidth=2, label=f'True Mean = {true_mean:.2f}')
axes[0].fill_between(sample_sizes, true_mean - 5, true_mean + 5, alpha=0.1, color='red')
axes[0].set_xlabel('Number of Samples (log scale)')
axes[0].set_ylabel('Sample Mean')
axes[0].set_title('Law of Large Numbers: Convergence', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

errors = np.abs(sample_means - true_mean)
axes[1].loglog(sample_sizes, errors, 'b-', alpha=0.7, linewidth=1, label='|Sample Mean - True Mean|')
theoretical_rate = sample_stds[0] * np.sqrt(sample_sizes[0]) / np.sqrt(sample_sizes)
axes[1].loglog(sample_sizes, theoretical_rate, 'r--', linewidth=2, label=r'Theoretical $O(1/\sqrt{n})$')
axes[1].set_xlabel('Number of Samples (log scale)')
axes[1].set_ylabel('Absolute Error (log scale)')
axes[1].set_title('Convergence Rate', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("\n\u2192 The sample mean converges to the true mean as n \u2192 \u221e")
print("  This is the foundation of Monte Carlo methods in RL!")

---

## 🔀 Section 3: Conditional Probability and Bayes' Theorem

### Conditional Probability

The probability of event $A$ given that event $B$ has occurred:

$$P(A|B) = \frac{P(A \cap B)}{P(B)}$$

### Bayes' Theorem

Allows us to **invert** conditional probabilities — go from effect to cause:

$$P(A|B) = \frac{P(B|A) \, P(A)}{P(B)}$$

Where:
- $P(A)$ = **prior** (what we believe before seeing evidence)
- $P(B|A)$ = **likelihood** (probability of evidence given hypothesis)
- $P(B)$ = **marginal likelihood** (normalizing constant)
- $P(A|B)$ = **posterior** (updated belief after seeing evidence)

### Law of Total Probability

If $A_1, A_2, \ldots, A_n$ partition $\Omega$:

$$P(B) = \sum_{i=1}^{n} P(B|A_i) \, P(A_i)$$

### Chain Rule

$$P(A_1, A_2, \ldots, A_n) = P(A_1) \, P(A_2|A_1) \, P(A_3|A_1,A_2) \cdots P(A_n|A_1,\ldots,A_{n-1})$$

### 🖼️ Image Processing Connection
- **Question:** Given a bright pixel, is it more likely sky or ground?
- This is exactly what image segmentation does — classify regions based on conditional probabilities!

In [ ]:
# Section 3: Conditional Probability with Images
# Create a synthetic sky/ground image and compute conditional probabilities

np.random.seed(42)
height, width = 200, 300

# Create sky region (top half) - bright, blue-ish values
sky = np.random.normal(loc=180, scale=30, size=(height // 2, width))
sky = np.clip(sky, 0, 255).astype(np.uint8)

# Create ground region (bottom half) - darker, brown-ish values
ground = np.random.normal(loc=80, scale=25, size=(height // 2, width))
ground = np.clip(ground, 0, 255).astype(np.uint8)

# Combine into one image
scene = np.vstack([sky, ground])

# Define events
bright_threshold = 200
bright_mask = scene > bright_threshold

# Region masks
sky_mask = np.zeros_like(scene, dtype=bool)
sky_mask[:height // 2, :] = True
ground_mask = ~sky_mask

# Compute probabilities
n_total = scene.size
P_bright = bright_mask.sum() / n_total
P_sky = sky_mask.sum() / n_total
P_ground = ground_mask.sum() / n_total

# P(Bright | Sky) and P(Bright | Ground)
P_bright_given_sky = (bright_mask & sky_mask).sum() / sky_mask.sum()
P_bright_given_ground = (bright_mask & ground_mask).sum() / ground_mask.sum()

# P(Sky | Bright) using Bayes' theorem
P_sky_given_bright = (P_bright_given_sky * P_sky) / P_bright
P_ground_given_bright = (P_bright_given_ground * P_ground) / P_bright

# Verify with law of total probability
P_bright_total = P_bright_given_sky * P_sky + P_bright_given_ground * P_ground

print("=" * 60)
print("CONDITIONAL PROBABILITY: Sky vs Ground Classification")
print("=" * 60)
print(f"\nBrightness threshold: > {bright_threshold}")
print("\nPrior Probabilities:")
print(f"  P(Sky)    = {P_sky:.4f}")
print(f"  P(Ground) = {P_ground:.4f}")
print("\nLikelihoods:")
print(f"  P(Bright | Sky)    = {P_bright_given_sky:.4f}")
print(f"  P(Bright | Ground) = {P_bright_given_ground:.4f}")
print("\nMarginal (Law of Total Probability):")
print(f"  P(Bright) = P(B|S)P(S) + P(B|G)P(G) = {P_bright_total:.4f}")
print(f"  P(Bright) [direct count]             = {P_bright:.4f} \u2713")
print("\nPosterior (Bayes' Theorem):")
print(f"  P(Sky | Bright)    = {P_sky_given_bright:.4f}  ({P_sky_given_bright:.1%})")
print(f"  P(Ground | Bright) = {P_ground_given_bright:.4f}  ({P_ground_given_bright:.1%})")
print(f"\n\u2192 A bright pixel is {P_sky_given_bright/P_ground_given_bright:.1f}x more likely to be sky!")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(scene, cmap='gray', vmin=0, vmax=255)
axes[0].axhline(height // 2, color='red', linestyle='--', linewidth=2)
axes[0].text(10, height // 4, 'SKY', fontsize=14, color='yellow', fontweight='bold')
axes[0].text(10, 3 * height // 4, 'GROUND', fontsize=14, color='yellow', fontweight='bold')
axes[0].set_title('Synthetic Scene', fontsize=11)
axes[0].axis('off')

bins = np.linspace(0, 255, 50)
axes[1].hist(scene[:height//2].flatten(), bins=bins, density=True, alpha=0.6,
            color='skyblue', label='Sky pixels')
axes[1].hist(scene[height//2:].flatten(), bins=bins, density=True, alpha=0.6,
            color='brown', label='Ground pixels')
axes[1].axvline(bright_threshold, color='red', linestyle='--', linewidth=2, label=f'Bright > {bright_threshold}')
axes[1].set_xlabel('Pixel Intensity')
axes[1].set_ylabel('Density')
axes[1].set_title('Likelihood: P(intensity | region)', fontsize=11)
axes[1].legend()

categories = ['P(Sky|Bright)', 'P(Ground|Bright)']
posteriors = [P_sky_given_bright, P_ground_given_bright]
colors = ['skyblue', 'brown']
bars = axes[2].bar(categories, posteriors, color=colors, edgecolor='black', linewidth=1.2)
for bar, val in zip(bars, posteriors):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
               f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Posterior Probability')
axes[2].set_title("Bayes' Theorem Result", fontsize=11)
axes[2].set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# Section 3 (continued): Bayesian Classifier for Image Regions
# Classify each pixel as sky or ground using Bayes' theorem

# Fit Gaussian distributions to each region
sky_pixels = scene[:height//2].flatten().astype(np.float64)
ground_pixels = scene[height//2:].flatten().astype(np.float64)

sky_mean, sky_std = np.mean(sky_pixels), np.std(sky_pixels)
ground_mean, ground_std = np.mean(ground_pixels), np.std(ground_pixels)

# Prior probabilities (assume equal)
prior_sky = 0.5
prior_ground = 0.5

# Compute likelihood for all pixel values
x_range = np.linspace(0, 255, 256)
likelihood_sky = stats.norm.pdf(x_range, sky_mean, sky_std)
likelihood_ground = stats.norm.pdf(x_range, ground_mean, ground_std)

# Compute posterior using Bayes' theorem for each pixel value
marginal = likelihood_sky * prior_sky + likelihood_ground * prior_ground
posterior_sky = (likelihood_sky * prior_sky) / marginal
posterior_ground = (likelihood_ground * prior_ground) / marginal

# Classify each pixel in the image
all_pixels = scene.flatten().astype(np.float64)
pixel_likelihood_sky = stats.norm.pdf(all_pixels, sky_mean, sky_std)
pixel_likelihood_ground = stats.norm.pdf(all_pixels, ground_mean, ground_std)
pixel_marginal = pixel_likelihood_sky * prior_sky + pixel_likelihood_ground * prior_ground
pixel_posterior_sky = (pixel_likelihood_sky * prior_sky) / pixel_marginal

# Create classification map
classification = (pixel_posterior_sky > 0.5).reshape(scene.shape)
accuracy = (classification == sky_mask).mean()

print("=" * 60)
print("BAYESIAN CLASSIFIER FOR IMAGE SEGMENTATION")
print("=" * 60)
print("\nLearned Parameters:")
print(f"  Sky:    \u03bc = {sky_mean:.1f}, \u03c3 = {sky_std:.1f}")
print(f"  Ground: \u03bc = {ground_mean:.1f}, \u03c3 = {ground_std:.1f}")
print(f"\nClassification Accuracy: {accuracy:.1%}")

# Visualization: Prior, Likelihood, Posterior
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Row 1: Bayes components
axes[0, 0].bar(['Sky', 'Ground'], [prior_sky, prior_ground], color=['skyblue', 'brown'],
              edgecolor='black')
axes[0, 0].set_title('Prior P(class)', fontsize=11)
axes[0, 0].set_ylim(0, 1)

axes[0, 1].plot(x_range, likelihood_sky, 'b-', linewidth=2, label=f'Sky (\u03bc={sky_mean:.0f})')
axes[0, 1].plot(x_range, likelihood_ground, 'brown', linewidth=2, label=f'Ground (\u03bc={ground_mean:.0f})')
axes[0, 1].set_xlabel('Pixel Intensity')
axes[0, 1].set_ylabel('Likelihood')
axes[0, 1].set_title('Likelihood P(x | class)', fontsize=11)
axes[0, 1].legend()

axes[0, 2].plot(x_range, posterior_sky, 'b-', linewidth=2, label='P(Sky | x)')
axes[0, 2].plot(x_range, posterior_ground, 'brown', linewidth=2, label='P(Ground | x)')
axes[0, 2].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0, 2].set_xlabel('Pixel Intensity')
axes[0, 2].set_ylabel('Posterior')
axes[0, 2].set_title('Posterior P(class | x)', fontsize=11)
axes[0, 2].legend()

# Row 2: Classification results
axes[1, 0].imshow(scene, cmap='gray')
axes[1, 0].set_title('Original Image', fontsize=11)
axes[1, 0].axis('off')

axes[1, 1].imshow(pixel_posterior_sky.reshape(scene.shape), cmap='RdBu', vmin=0, vmax=1)
axes[1, 1].set_title('P(Sky | pixel value)', fontsize=11)
axes[1, 1].axis('off')

classification_vis = np.zeros((*scene.shape, 3), dtype=np.uint8)
classification_vis[classification] = [135, 206, 235]   # Sky blue
classification_vis[~classification] = [139, 90, 43]    # Brown
axes[1, 2].imshow(classification_vis)
axes[1, 2].set_title(f'Classification (Acc: {accuracy:.1%})', fontsize=11)
axes[1, 2].axis('off')

plt.tight_layout()
plt.show()

---

## 🔗 Section 4: The Markov Property

### Formal Definition

A stochastic process $\{X_t\}$ has the **Markov property** if:

$$P(X_{t+1} | X_t, X_{t-1}, \ldots, X_0) = P(X_{t+1} | X_t)$$

### Intuition: "Memorylessness"

The future depends **only on the present**, not on how we got here.

### Why This Matters for RL

The Markov property is the **fundamental assumption** of Reinforcement Learning:
- The **next state** depends only on the current state and action: $P(s_{t+1} | s_t, a_t)$
- We don't need to remember the entire history
- This makes the problem computationally tractable

### 🖼️ Image Processing Context

When applying image transformations:
- Applying a Gaussian blur to an image depends only on the **current pixel values**
- It doesn't matter if the image was previously sharpened, rotated, or cropped
- The result of `blur(image)` is the same regardless of editing history

$$\text{result} = \text{filter}(\text{current\_image})$$

This means image processing pipelines naturally form **Markov chains**!

In [ ]:
# Section 4: Visual Demonstration of the Markov Property
# Show that image transformations depend only on current state

from skimage.filters import gaussian, unsharp_mask
from skimage.exposure import adjust_gamma

# Load or create a base image
try:
    base_img = data.astronaut()
    base_img = color.rgb2gray(base_img)
except Exception:
    base_img = create_gaussian_blobs(256).astype(np.float64) / 255.0

if base_img.max() > 1.0:
    base_img = base_img.astype(np.float64) / 255.0

# Define transformations
def apply_blur(img):
    return gaussian(img, sigma=2)

def apply_sharpen(img):
    return unsharp_mask(img, radius=2, amount=1.5)

def apply_gamma(img):
    return adjust_gamma(img, gamma=0.7)

# Path 1: Original -> Blur -> Sharpen -> Gamma
path1_step1 = apply_blur(base_img)
path1_step2 = apply_sharpen(path1_step1)
path1_step3 = apply_gamma(path1_step2)

# Path 2: Original -> Gamma -> Blur -> Sharpen -> Gamma
# Different history, but arrive at same intermediate state then apply gamma
path2_step1 = apply_gamma(base_img)  # Different first step
path2_intermediate = apply_sharpen(apply_blur(base_img))  # Same state as path1_step2
path2_step3 = apply_gamma(path2_intermediate)  # Same operation on same state

# The Markov property: same state + same action = same result
difference = np.abs(path1_step3 - path2_step3).max()

print("=" * 60)
print("MARKOV PROPERTY IN IMAGE PROCESSING")
print("=" * 60)
print("\nPath 1: Original \u2192 Blur \u2192 Sharpen \u2192 Gamma")
print("Path 2: (different history) \u2192 same Sharpen result \u2192 Gamma")
print(f"\nMax difference between results: {difference:.10f}")
print(f"Results are identical: {difference < 1e-10} \u2713")
print("\n\u2192 The Markov property holds! The result depends only on")
print("  the current state, not on how we arrived there.")

# Visualization: Image transformation chain
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

# Path 1
path1_images = [base_img, path1_step1, path1_step2, path1_step3]
path1_labels = ['Original\n(State $s_0$)', 'After Blur\n(State $s_1$)',
                'After Sharpen\n(State $s_2$)', 'After Gamma\n(State $s_3$)']

for i, (img_show, label) in enumerate(zip(path1_images, path1_labels)):
    axes[0, i].imshow(img_show, cmap='gray', vmin=0, vmax=1)
    axes[0, i].set_title(label, fontsize=10)
    axes[0, i].axis('off')
    if i > 0:
        axes[0, i].annotate('', xy=(-0.1, 0.5), xytext=(-0.2, 0.5),
                           xycoords='axes fraction', textcoords='axes fraction',
                           arrowprops=dict(arrowstyle='->', color='green', lw=2))

axes[0, 0].set_ylabel('Path 1', fontsize=12, fontweight='bold')

# Path 2
path2_images = [base_img, path2_step1, path2_intermediate, path2_step3]
path2_labels = ['Original\n(State $s_0$)', 'After Gamma\n(Different!)',
                'Blur+Sharpen\n(Same as $s_2$!)', 'After Gamma\n(Same as $s_3$!)']

for i, (img_show, label) in enumerate(zip(path2_images, path2_labels)):
    axes[1, i].imshow(img_show, cmap='gray', vmin=0, vmax=1)
    axes[1, i].set_title(label, fontsize=10)
    axes[1, i].axis('off')

axes[1, 0].set_ylabel('Path 2', fontsize=12, fontweight='bold')

fig.suptitle('Markov Property: Same State + Same Action = Same Result\n(regardless of history)',
            fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Section 4 (continued): Markov Chain Simulation
# States represent image quality levels; transitions depend on actions

# Define states: image quality levels
states = ['Poor', 'Fair', 'Good', 'Excellent']
n_states = len(states)

# Transition matrix: P(next_state | current_state)
# Rows = current state, Columns = next state
transition_matrix = np.array([
    [0.3, 0.5, 0.15, 0.05],   # From Poor
    [0.1, 0.3, 0.45, 0.15],   # From Fair
    [0.05, 0.1, 0.4, 0.45],   # From Good
    [0.02, 0.08, 0.3, 0.6],   # From Excellent
])

# Verify rows sum to 1
assert np.allclose(transition_matrix.sum(axis=1), 1.0)

# Simulate the Markov chain
n_steps = 500
current_state = 0  # Start in 'Poor' state
state_trajectory = [current_state]

np.random.seed(42)
for _ in range(n_steps - 1):
    next_state = np.random.choice(n_states, p=transition_matrix[current_state])
    state_trajectory.append(next_state)
    current_state = next_state

state_trajectory = np.array(state_trajectory)

# Compute stationary distribution (eigenvector method)
eigenvalues, eigenvectors = np.linalg.eig(transition_matrix.T)
stationary_idx = np.argmin(np.abs(eigenvalues - 1.0))
stationary_dist = np.real(eigenvectors[:, stationary_idx])
stationary_dist = stationary_dist / stationary_dist.sum()

# Empirical distribution from simulation
empirical_dist = np.array([(state_trajectory == i).mean() for i in range(n_states)])

print("=" * 60)
print("MARKOV CHAIN: Image Quality State Transitions")
print("=" * 60)
print("\nTransition Matrix P(next | current):")
print(f"{'':>12}", end='')
for s in states:
    print(f"{s:>10}", end='')
print()
for i, s in enumerate(states):
    print(f"{s:>12}", end='')
    for j in range(n_states):
        print(f"{transition_matrix[i,j]:>10.2f}", end='')
    print()
print("\nStationary Distribution (long-run behavior):")
print(f"  {'State':<12} {'Theoretical':>12} {'Empirical':>12}")
print(f"  {'-'*38}")
for i, s in enumerate(states):
    print(f"  {s:<12} {stationary_dist[i]:>12.4f} {empirical_dist[i]:>12.4f}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# State trajectory
axes[0].plot(state_trajectory[:100], 'b-', alpha=0.7, linewidth=1)
axes[0].scatter(range(100), state_trajectory[:100], c=state_trajectory[:100],
               cmap='RdYlGn', s=15, zorder=5)
axes[0].set_yticks(range(n_states))
axes[0].set_yticklabels(states)
axes[0].set_xlabel('Time Step')
axes[0].set_title('Markov Chain Trajectory (first 100 steps)', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Stationary vs empirical distribution
x_pos = np.arange(n_states)
width = 0.35
axes[1].bar(x_pos - width/2, stationary_dist, width, label='Theoretical', color='steelblue')
axes[1].bar(x_pos + width/2, empirical_dist, width, label='Empirical', color='coral')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(states)
axes[1].set_ylabel('Probability')
axes[1].set_title('Stationary Distribution', fontsize=11)
axes[1].legend()

# Convergence of state probabilities over time
running_probs = np.zeros((n_steps, n_states))
for t in range(1, n_steps):
    for i in range(n_states):
        running_probs[t, i] = (state_trajectory[:t+1] == i).mean()

for i, s in enumerate(states):
    axes[2].plot(running_probs[1:, i], linewidth=1.5, label=s)
    axes[2].axhline(stationary_dist[i], linestyle='--', alpha=0.5, color=f'C{i}')

axes[2].set_xlabel('Time Step')
axes[2].set_ylabel('Running Proportion')
axes[2].set_title('Convergence to Stationary Distribution', fontsize=11)
axes[2].legend(loc='right', fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 🌐 Section 5: Connecting Probability to Images

### Images ARE Probability Distributions

A normalized histogram of pixel intensities is a valid probability distribution:
- It satisfies all Kolmogorov axioms
- It completely characterizes the first-order statistics of the image

### Joint Distributions for Color Images

For an RGB image, we can study the **joint distribution** $P(R, G, B)$:
- Marginals: $P(R)$, $P(G)$, $P(B)$ (individual channel histograms)
- Joint: $P(R, G)$ (2D histogram showing how channels correlate)

### Image Entropy

Shannon entropy measures the **information content** of an image:

$$H(X) = -\sum_{x} p(x) \log_2 p(x)$$

- **High entropy** → complex, unpredictable image (e.g., noise)
- **Low entropy** → simple, predictable image (e.g., solid color)
- Maximum entropy for 8-bit image: $H_{max} = \log_2(256) = 8$ bits/pixel

In [ ]:
# Section 5: Image Entropy Comparison

def compute_entropy(image):
    """Compute Shannon entropy of an image."""
    hist, _ = np.histogram(image.flatten(), bins=256, range=(0, 255))
    pmf = hist / hist.sum()
    pmf = pmf[pmf > 0]  # Remove zeros to avoid log(0)
    return -np.sum(pmf * np.log2(pmf))

# Create different types of images
np.random.seed(42)

# 1. Uniform random noise (maximum entropy)
noise_img = np.random.randint(0, 256, size=(256, 256), dtype=np.uint8)

# 2. Natural image
try:
    natural_img = data.camera()
except Exception:
    natural_img = create_gaussian_blobs(256)

# 3. Low entropy image (mostly constant with slight variation)
constant_img = np.full((256, 256), 128, dtype=np.uint8)
constant_img += np.random.randint(-2, 3, size=(256, 256), dtype=np.int8).view(np.uint8)
constant_img = np.clip(constant_img.astype(np.int16), 0, 255).astype(np.uint8)

# 4. Gradient image
gradient_img = np.tile(np.linspace(0, 255, 256).astype(np.uint8), (256, 1))

images = [noise_img, natural_img, gradient_img, constant_img]
titles = ['Uniform Noise', 'Natural Image', 'Gradient', 'Near-Constant']
entropies = [compute_entropy(img) for img in images]

print("=" * 60)
print("IMAGE ENTROPY COMPARISON")
print("=" * 60)
print(f"\nMaximum possible entropy (8-bit): {np.log2(256):.4f} bits/pixel")
print(f"\n{'Image Type':<20} {'Entropy (bits)':>15} {'% of Maximum':>15}")
print("-" * 52)
for title, entropy in zip(titles, entropies):
    print(f"{title:<20} {entropy:>15.4f} {entropy/8*100:>14.1f}%")

# Visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for i, (img_show, title, entropy) in enumerate(zip(images, titles, entropies)):
    axes[0, i].imshow(img_show, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(f'{title}\nH = {entropy:.2f} bits', fontsize=10)
    axes[0, i].axis('off')
    
    axes[1, i].hist(img_show.flatten(), bins=256, range=(0, 255),
                   density=True, color='steelblue', alpha=0.7)
    axes[1, i].set_xlabel('Pixel Value')
    axes[1, i].set_ylabel('P(x)')
    axes[1, i].set_xlim(0, 255)

plt.tight_layout()
plt.show()

print("\n\u2192 Higher entropy = more information = more complex image")
print("  An RL agent needs more capacity to process high-entropy images!")

In [ ]:
# Section 5 (continued): Joint Probability Distribution of Color Channels

# Load a color image
try:
    color_img = data.astronaut()  # RGB image
except Exception:
    # Fallback: create a synthetic color image
    np.random.seed(42)
    color_img = np.zeros((256, 256, 3), dtype=np.uint8)
    color_img[:128, :, 0] = np.random.normal(180, 30, (128, 256)).clip(0, 255).astype(np.uint8)
    color_img[:128, :, 2] = np.random.normal(200, 20, (128, 256)).clip(0, 255).astype(np.uint8)
    color_img[128:, :, 1] = np.random.normal(150, 25, (128, 256)).clip(0, 255).astype(np.uint8)
    color_img[128:, :, 0] = np.random.normal(100, 20, (128, 256)).clip(0, 255).astype(np.uint8)

R = color_img[:, :, 0].flatten()
G = color_img[:, :, 1].flatten()
B = color_img[:, :, 2].flatten()

# Compute joint distribution (2D histogram) for R and G
joint_hist, xedges, yedges = np.histogram2d(R, G, bins=64, range=[[0, 255], [0, 255]])
joint_prob = joint_hist / joint_hist.sum()

# Compute marginals from joint
marginal_R = joint_prob.sum(axis=1)
marginal_G = joint_prob.sum(axis=0)

# Correlation coefficient
correlation = np.corrcoef(R, G)[0, 1]

print("=" * 60)
print("JOINT PROBABILITY DISTRIBUTION: R and G Channels")
print("=" * 60)
print(f"\nImage shape: {color_img.shape}")
print(f"Total pixels: {len(R)}")
print("\nMarginal Statistics:")
print(f"  Red:   E[R] = {np.mean(R):.1f}, Std[R] = {np.std(R):.1f}")
print(f"  Green: E[G] = {np.mean(G):.1f}, Std[G] = {np.std(G):.1f}")
print(f"  Blue:  E[B] = {np.mean(B):.1f}, Std[B] = {np.std(B):.1f}")
print(f"\nCorrelation \u03c1(R, G) = {correlation:.4f}")
print(f"Joint distribution shape: {joint_prob.shape}")
print(f"Joint distribution sums to: {joint_prob.sum():.6f}")

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(color_img)
axes[0].set_title('Original Color Image', fontsize=11)
axes[0].axis('off')

# Marginal distributions
axes[1].hist(R, bins=64, range=(0, 255), density=True, alpha=0.6, color='red', label='R')
axes[1].hist(G, bins=64, range=(0, 255), density=True, alpha=0.6, color='green', label='G')
axes[1].hist(B, bins=64, range=(0, 255), density=True, alpha=0.6, color='blue', label='B')
axes[1].set_xlabel('Pixel Value')
axes[1].set_ylabel('Density')
axes[1].set_title('Marginal Distributions', fontsize=11)
axes[1].legend()

# Joint distribution as heatmap
im = axes[2].imshow(joint_prob.T, origin='lower', aspect='auto',
                    extent=[0, 255, 0, 255], cmap='hot')
axes[2].set_xlabel('Red Channel')
axes[2].set_ylabel('Green Channel')
axes[2].set_title(f'Joint P(R, G)\n\u03c1 = {correlation:.3f}', fontsize=11)
plt.colorbar(im, ax=axes[2], fraction=0.046)

# Scatter plot (subsampled)
subsample = np.random.choice(len(R), size=5000, replace=False)
axes[3].scatter(R[subsample], G[subsample], alpha=0.1, s=1, c='purple')
axes[3].set_xlabel('Red Channel')
axes[3].set_ylabel('Green Channel')
axes[3].set_title('R vs G Scatter (5000 samples)', fontsize=11)
axes[3].set_xlim(0, 255)
axes[3].set_ylim(0, 255)

plt.tight_layout()
plt.show()

---

## 🎮 Section 6: Why Probability Matters for RL

### The RL-Probability Connection

Every component of RL is built on probability theory:

| RL Concept | Probability Foundation |
|------------|----------------------|
| **Policy** | Probability distribution over actions: $\pi(a|s)$ |
| **Value Function** | Expectation of returns: $V^\pi(s) = E_\pi\left[\sum_{t=0}^{\infty} \gamma^t R_t \mid S_0 = s\right]$ |
| **State Transitions** | Conditional probability: $P(s'|s, a)$ |
| **Rewards** | May be stochastic: $R(s, a) \sim P(r|s, a)$ |
| **Exploration** | Sampling from distributions |

### For Image Processing RL:

- **State $s$:** The current image (pixel distribution)
- **Action $a$:** Image enhancement operation (brighten, sharpen, denoise, etc.)
- **Policy $\pi(a|s)$:** Given current image quality, what action to take?
- **Transition $P(s'|s, a)$:** How the image changes after applying an action
- **Reward $R$:** Image quality improvement metric

In [ ]:
# Section 6: Stochastic Policy for Image Enhancement

# Define a simple image enhancement scenario
actions = ['Brighten', 'Sharpen', 'Denoise']
n_actions = len(actions)

# Define states based on image characteristics
image_states = {
    'dark_blurry': {'description': 'Dark and blurry image', 'brightness': 0.3, 'sharpness': 0.2, 'noise': 0.4},
    'dark_sharp': {'description': 'Dark but sharp image', 'brightness': 0.3, 'sharpness': 0.8, 'noise': 0.3},
    'bright_noisy': {'description': 'Bright but noisy image', 'brightness': 0.8, 'sharpness': 0.5, 'noise': 0.8},
    'good_quality': {'description': 'Already good quality', 'brightness': 0.7, 'sharpness': 0.7, 'noise': 0.2},
}

# Define policy: pi(action | state) - probability of each action given state
# A smart policy should prefer actions that address the image's main problem
policy = {
    'dark_blurry':  [0.50, 0.35, 0.15],  # Mostly brighten, some sharpen
    'dark_sharp':   [0.75, 0.10, 0.15],  # Mostly brighten
    'bright_noisy': [0.05, 0.15, 0.80],  # Mostly denoise
    'good_quality': [0.33, 0.34, 0.33],  # Uniform - already good
}

# Simulate action sampling from the policy
np.random.seed(42)
n_episodes = 1000

print("=" * 60)
print("STOCHASTIC POLICY FOR IMAGE ENHANCEMENT")
print("=" * 60)
print(f"\nActions available: {actions}")
print("\nPolicy \u03c0(a|s):")
print(f"{'State':<16}", end='')
for a in actions:
    print(f"{a:>12}", end='')
print()
print("-" * 52)
for state_name, probs in policy.items():
    print(f"{state_name:<16}", end='')
    for p in probs:
        print(f"{p:>12.2f}", end='')
    print()

# Sample actions for a specific state
test_state = 'dark_blurry'
action_probs = policy[test_state]
sampled_actions = np.random.choice(n_actions, size=n_episodes, p=action_probs)

# Count action frequencies
action_counts = np.bincount(sampled_actions, minlength=n_actions)
action_freqs = action_counts / n_episodes

print(f"\n\nSimulation: Sampling {n_episodes} actions from \u03c0(a|s='{test_state}')")
print(f"{'Action':<12} {'Policy \u03c0(a|s)':>14} {'Empirical Freq':>16} {'Count':>8}")
print("-" * 52)
for i, a in enumerate(actions):
    print(f"{a:<12} {action_probs[i]:>14.3f} {action_freqs[i]:>16.3f} {action_counts[i]:>8}")

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Policy heatmap
policy_matrix = np.array(list(policy.values()))
im = axes[0].imshow(policy_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
axes[0].set_xticks(range(n_actions))
axes[0].set_xticklabels(actions, rotation=30)
axes[0].set_yticks(range(len(policy)))
axes[0].set_yticklabels(list(policy.keys()))
axes[0].set_title('Policy \u03c0(a|s) Heatmap', fontsize=11)
for i in range(len(policy)):
    for j in range(n_actions):
        axes[0].text(j, i, f'{policy_matrix[i, j]:.2f}',
                    ha='center', va='center', fontsize=10,
                    color='white' if policy_matrix[i, j] > 0.5 else 'black')
plt.colorbar(im, ax=axes[0], fraction=0.046)

# Action distribution for test state
x_pos = np.arange(n_actions)
width = 0.35
axes[1].bar(x_pos - width/2, action_probs, width, label='Policy \u03c0(a|s)',
           color='steelblue', edgecolor='black')
axes[1].bar(x_pos + width/2, action_freqs, width, label=f'Empirical (n={n_episodes})',
           color='coral', edgecolor='black')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(actions)
axes[1].set_ylabel('Probability')
axes[1].set_title(f"Action Distribution for '{test_state}'", fontsize=11)
axes[1].legend()
axes[1].set_ylim(0, 1)

# Cumulative action selection over episodes
cumulative = np.zeros((n_episodes, n_actions))
for t in range(n_episodes):
    for i in range(n_actions):
        cumulative[t, i] = (sampled_actions[:t+1] == i).mean()

for i, a in enumerate(actions):
    axes[2].plot(cumulative[:, i], linewidth=1.5, label=a)
    axes[2].axhline(action_probs[i], linestyle='--', alpha=0.5, color=f'C{i}')

axes[2].set_xlabel('Episode')
axes[2].set_ylabel('Running Action Proportion')
axes[2].set_title('Policy Convergence Over Episodes', fontsize=11)
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n\u2192 A stochastic policy allows the agent to EXPLORE different actions")
print("  while still favoring actions that are likely to improve the image.")
print("  This exploration-exploitation trade-off is central to RL!")

---

## 📝 Summary

### Key Takeaways

| Concept | Definition | RL Connection |
|---------|-----------|---------------|
| **Probability Space** | $(\Omega, \mathcal{F}, P)$ with Kolmogorov axioms | State/action spaces in RL |
| **Random Variable** | $X: \Omega \to \mathbb{R}$ with PMF $p(x)$ | Rewards, state features |
| **Expectation** | $E[X] = \sum_x x \cdot p(x)$ | Value functions $V(s)$, $Q(s,a)$ |
| **Variance** | $\text{Var}(X) = E[X^2] - (E[X])^2$ | Risk-sensitive RL |
| **Conditional Probability** | $P(A|B) = P(A \cap B) / P(B)$ | $P(s'|s,a)$, $\pi(a|s)$ |
| **Bayes' Theorem** | $P(A|B) = P(B|A)P(A)/P(B)$ | Belief updates, POMDPs |
| **Markov Property** | $P(X_{t+1}|X_t,...,X_0) = P(X_{t+1}|X_t)$ | Foundation of MDPs |
| **Entropy** | $H(X) = -\sum p(x)\log_2 p(x)$ | Exploration bonuses, max-entropy RL |

### 🚀 What's Next?

**Module 3.2: Markov Decision Processes** — We'll build on the Markov property to define the full RL framework with states, actions, transitions, and rewards.

---

*📚 Further Reading:*
- Bertsekas & Tsitsiklis, "Introduction to Probability" (2008)
- Sutton & Barto, "Reinforcement Learning: An Introduction" Chapter 3
- Cover & Thomas, "Elements of Information Theory" (for entropy)